# PPO Continuous State and Reward Lab

This notebook runs PPO continuous baselines while keeping the editable experiment definition in a few visible cells. The intended edit points are the state-feature list, the reward weights/scales, and the usual training knobs. The training code itself stays in the package and the notebook writes JSON spec files so each run can be repeated later.


## 1. Setup

Run this first. It finds the project root, imports the notebook helpers, and loads the state/reward spec builders used by the subprocess runner.


In [ ]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

import numpy as np

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    for root in (candidate, candidate / 'TeleopWithRL'):
        if (root / 'matlab_literal_env').exists() and (root / 'notebooks' / '_teleop_nb.py').exists():
            for path_to_add in (root.parent, root):
                if str(path_to_add) not in sys.path:
                    sys.path.insert(0, str(path_to_add))
            break
    else:
        continue
    break
else:
    raise RuntimeError('Could not find TeleopWithRL notebook root.')

from notebooks._teleop_nb import load_json, project_python_executable, repo_paths, show_image, show_rows
from TeleopWithRL import config as cfg
from TeleopWithRL.matlab_literal_env.policy_gradient_experiments.paths import suite_root as policy_gradient_suite_root
from TeleopWithRL.matlab_literal_env.studies.common import save_json
from TeleopWithRL.matlab_literal_env.studies.dqn_state_variants import (
    available_custom_state_feature_rows,
    build_custom_dqn_state_variant_from_spec,
    get_dqn_state_variant,
)
from TeleopWithRL.matlab_literal_env.studies.rewarding import (
    DEFAULT_ACTION_DELTA_SCALE_V,
    DEFAULT_ACTION_SCALE_V,
    DEFAULT_FORCE_DIFF_SCALE_N,
    DEFAULT_TRACKING_SCALE_M,
    DEFAULT_TRANSPARENCY_SCALE_W,
    DEFAULT_VELOCITY_ERROR_SCALE_MPS,
    compute_reward_terms,
    reward_formula_from_context,
    reward_variant_from_name,
    reward_variant_from_spec,
)

P = repo_paths()
REPO = P['repo']
WORKSPACE = REPO.parent
PYTHON = project_python_executable(REPO)
PG_RESULTS = REPO / 'matlab_literal_env' / 'policy_gradient_experiments' / 'results'

def short_hash(payload: dict) -> str:
    raw = json.dumps(payload, sort_keys=True, default=str).encode('utf-8')
    return hashlib.sha1(raw).hexdigest()[:8]


## 2. Experiment Knobs

These are the normal run settings. For quick smoke tests, reduce `train_episodes`, `parallel_envs`, and `test_episodes`; for serious comparisons, keep the seed fixed and change one idea at a time.


In [ ]:
ALGO_KEY = 'ppo_continuous'
ALGO_LABEL = 'PPO Continuous'
ALGO_TAG = 'ppo'
RUN_DIR = 'ppo'
FINAL_RESULTS_SUBFOLDER = 'ppoFinalModel'

CFG = {
    'experiment_label': 'state_reward_lab01',
    'env_mode': 'changing_skin_fat',
    'episode_duration_s': 30.0,
    'env_switch_time_s': 10.0,
    'reset_position_mode': 'midpoint',
    'stroke_limit_mode': 'clamp',
    'force_amp_N': 5.0,
    'force_bias_N': 15.0,
    'force_freq_rad_s': 6.0,
    'force_phase_rad': 0.0,
    'force_waveform': 'sine',
    'train_episodes': 334,
    'total_timesteps': 500_000,
    'parallel_envs': 8,
    'vec_env': 'subproc',
    'ppo_n_steps': 256,
    'ppo_batch_size': 512,
    'ppo_n_epochs': 4,
    'ppo_device': 'auto',
    'eval_every_episodes': 150,
    'test_episodes': 32,
    'seed': 42,
    'parallel_workers': 1,
    'worker_torch_threads': 1,
    'skip_existing': False,
}

FE_MODE = 'switched_dynamics'
FE_KEY = 'dyn'
MODE_LABELS = {
    FE_KEY: FE_MODE,
}


## 3. State Space

This is the main state-space cell. The default is a full physical MDP-style state: master/slave positions, velocities, chamber pressures, line mass flows, valve states, forces, previous action, and time/context. Change `True` to `False` to remove a variable, or turn on one of the derived variables near the bottom.


In [ ]:
USE_STATE_FEATURE = {
    # Physical plant state: positions and velocities
    'x_m': True,
    'x_s': True,
    'v_m': True,
    'v_s': True,

    # Physical plant state: chamber pressures
    'P_m1': False,
    'P_m2': False,
    'P_s1': False,
    'P_s2': False,

    # Physical plant state: tube mass-flow states
    'mdot_L1': False,
    'mdot_L2': False,

    # Physical plant state: valve dynamics
    'x_v': False,
    'x_v_dot': False,

    # Exogenous inputs, measured forces, and context
    'F_h': False,
    'F_e': False,
    'u_v': False,
    'env_id': False,
    'time_fraction': False,

    # Optional action/context variables
    'requested_u_v': False,

    # Optional derived variables. These are redundant if the raw variables above are present,
    # but they can help smaller networks by giving common errors directly.
    'tracking_error': False,
    'velocity_error': False,
    'transparency_error': False,
    'force_diff': False,
    'delta_P_m': False,
    'delta_P_s': False,
    'P_m1_minus_P_s1': False,
    'P_m2_minus_P_s2': False,

    # Optional equilibrium-centered positions. Usually choose these OR absolute x_m/x_s.
    'x_m_eq': False,
    'x_s_eq': False,
    'x_m_centered': False,
    'x_s_centered': False,
}

ALL_STATE_VARIABLES = available_custom_state_feature_rows()
state_lookup = {row['feature']: row for row in ALL_STATE_VARIABLES}
unknown_features = [feature for feature in USE_STATE_FEATURE if feature not in state_lookup]
if unknown_features:
    raise KeyError(f'Unknown state feature(s): {unknown_features}')

def state_feature_group(feature):
    if feature in {'x_m', 'x_s', 'x_m_eq', 'x_s_eq', 'x_m_centered', 'x_s_centered', 'tracking_error'}:
        return 'position'
    if feature in {'v_m', 'v_s', 'velocity_error'}:
        return 'velocity'
    if feature in {'P_m1', 'P_m2', 'P_s1', 'P_s2', 'delta_P_m', 'delta_P_s', 'P_m1_minus_P_s1', 'P_m2_minus_P_s2'}:
        return 'pressure'
    if feature in {'mdot_L1', 'mdot_L2'}:
        return 'mass_flow'
    if feature in {'x_v', 'x_v_dot'}:
        return 'valve'
    if feature in {'F_h', 'F_e', 'force_diff', 'transparency_error'}:
        return 'force_transparency'
    if feature in {'u_v', 'requested_u_v'}:
        return 'action_memory'
    if feature in {'env_id', 'time_fraction'}:
        return 'context'
    return 'other'

catalog_rows = []
for row in ALL_STATE_VARIABLES:
    feature = row['feature']
    display_row = dict(row)
    display_row['group'] = state_feature_group(feature)
    display_row['selected'] = bool(USE_STATE_FEATURE.get(feature, False))
    catalog_rows.append(display_row)
show_rows(catalog_rows, title='Full MDP variable catalog', max_rows=80)

STATE_VARIANT = get_dqn_state_variant('S11_absolute_posvel_only')
SELECTED_STATE_FEATURES = list(STATE_VARIANT.feature_names)

STATE_SPEC = {
    'name': STATE_VARIANT.name,
    'description': STATE_VARIANT.description,
    'selected_features': SELECTED_STATE_FEATURES,
}

selected_state_rows = []
for idx, feature in enumerate(STATE_VARIANT.feature_names, start=1):
    row = dict(state_lookup[feature])
    row['order'] = idx
    row['group'] = state_feature_group(feature)
    selected_state_rows.append(row)

show_rows(selected_state_rows, title=f'Selected state space: {STATE_VARIANT.name}', max_rows=80)
print(f'Observation dimension: {STATE_VARIANT.obs_dim}')


## 4. Action Space

PPO continuous keeps the same action interface: one continuous valve-voltage command clipped to the environment voltage range.


In [ ]:
ACTION_SPACE = {
    'action': 'u_v',
    'type': 'continuous voltage',
    'low_v': float(np.min(cfg.V_LEVELS)),
    'high_v': float(np.max(cfg.V_LEVELS)),
    'scale_used_for_state_and_reward': DEFAULT_ACTION_SCALE_V,
}
show_rows([ACTION_SPACE], title='Action space used by PPO continuous', max_rows=5)


## 5. Reward Function

This cell controls the **structure** and **normalization** of the reward. Each row in `REWARD_TERMS` chooses what signal to use, how to shape it, and which named scale normalizes it. To change normalization, edit `scale_name` or add a new entry to `REWARD_SCALE_CATALOG`.


In [ ]:
VALVE_POSITION_SCALE = max(abs(float(cfg.KV)) * DEFAULT_ACTION_SCALE_V, 1e-9)
VALVE_VELOCITY_SCALE = max(150.0 * VALVE_POSITION_SCALE, 1e-9)

REWARD_SOURCE_CATALOG = [
    {'source': 'pos_error', 'meaning': 'x_m - x_s tracking error [m]', 'suggested_scale': 'tracking_error_limit_m'},
    {'source': 'velocity_error', 'meaning': 'v_m - v_s [m/s]', 'suggested_scale': 'velocity_error_mps'},
    {'source': 'transparency_error', 'meaning': '(F_e * v_m) - (F_h * v_s) [W]', 'suggested_scale': 'power_error_practical_w'},
    {'source': 'force_diff', 'meaning': 'F_e - F_h [N]', 'suggested_scale': 'force_difference_n'},
    {'source': 'u_v', 'meaning': 'applied valve voltage [V]', 'suggested_scale': 'action_voltage_v'},
    {'source': 'action_delta', 'meaning': 'u_v - previous_u_v [V]', 'suggested_scale': 'action_delta_voltage_v'},
    {'source': 'F_h', 'meaning': 'human/master force input [N]', 'suggested_scale': 'human_force_est_n'},
    {'source': 'F_e', 'meaning': 'environment/slave force [N]', 'suggested_scale': 'environment_force_theoretical_n'},
    {'source': 'x_m', 'meaning': 'absolute master position [m]', 'suggested_scale': 'half_stroke_m'},
    {'source': 'x_s', 'meaning': 'absolute slave position [m]', 'suggested_scale': 'half_stroke_m'},
    {'source': 'x_m_centered', 'meaning': 'master position relative to reset equilibrium [m]', 'suggested_scale': 'half_stroke_m'},
    {'source': 'x_s_centered', 'meaning': 'slave position relative to reset equilibrium [m]', 'suggested_scale': 'half_stroke_m'},
    {'source': 'v_m', 'meaning': 'master velocity [m/s]', 'suggested_scale': 'obs_velocity_mps'},
    {'source': 'v_s', 'meaning': 'slave velocity [m/s]', 'suggested_scale': 'obs_velocity_mps'},
    {'source': 'P_m1', 'meaning': 'master chamber 1 pressure [Pa]', 'suggested_scale': 'pressure_supply_pa'},
    {'source': 'P_m2', 'meaning': 'master chamber 2 pressure [Pa]', 'suggested_scale': 'pressure_supply_pa'},
    {'source': 'P_s1', 'meaning': 'slave chamber 1 pressure [Pa]', 'suggested_scale': 'pressure_supply_pa'},
    {'source': 'P_s2', 'meaning': 'slave chamber 2 pressure [Pa]', 'suggested_scale': 'pressure_supply_pa'},
    {'source': 'delta_P_m', 'meaning': 'P_m1 - P_m2 [Pa]', 'suggested_scale': 'pressure_difference_pa'},
    {'source': 'delta_P_s', 'meaning': 'P_s1 - P_s2 [Pa]', 'suggested_scale': 'pressure_difference_pa'},
    {'source': 'P_m1_minus_P_s1', 'meaning': 'cross-piston chamber 1 pressure error [Pa]', 'suggested_scale': 'pressure_difference_pa'},
    {'source': 'P_m2_minus_P_s2', 'meaning': 'cross-piston chamber 2 pressure error [Pa]', 'suggested_scale': 'pressure_difference_pa'},
    {'source': 'mdot_L1', 'meaning': 'tube mass-flow state 1 [kg/s]', 'suggested_scale': 'mass_flow_kg_s'},
    {'source': 'mdot_L2', 'meaning': 'tube mass-flow state 2 [kg/s]', 'suggested_scale': 'mass_flow_kg_s'},
    {'source': 'x_v', 'meaning': 'valve spool position', 'suggested_scale': 'valve_position'},
    {'source': 'x_v_dot', 'meaning': 'valve spool velocity', 'suggested_scale': 'valve_velocity'},
    {'source': 'edge_severity', 'meaning': '0 away from stroke edge, 1 at edge; uses edge_buffer_m', 'suggested_scale': 'unit_interval'},
    {'source': 'low_force_edge_severity', 'meaning': 'edge severity multiplied by low-force factor', 'suggested_scale': 'unit_interval'},
    {'source': 'time', 'meaning': 'episode time [s]', 'suggested_scale': 'episode_duration_s'},
    {'source': 'time_fraction', 'meaning': 'episode progress from 0 to 1', 'suggested_scale': 'unit_interval'},
    {'source': 'env_id', 'meaning': 'skin=0, fat=1', 'suggested_scale': 'unit_interval'},
]

REWARD_SCALE_CATALOG = {
    # Dimensionless/context scales
    'unit_interval': {'value': 1.0, 'unit': '-', 'use_for': 'edge_severity, time_fraction, env_id'},
    'episode_duration_s': {'value': float(CFG['episode_duration_s']), 'unit': 's', 'use_for': 'time'},
    'env_switch_time_s': {'value': float(CFG['env_switch_time_s']), 'unit': 's', 'use_for': 'pre/post switch timing terms'},

    # Position scales
    'stroke_m': {'value': float(cfg.L_CYL), 'unit': 'm', 'use_for': 'absolute x_m/x_s over full cylinder stroke'},
    'half_stroke_m': {'value': float(cfg.OBS_SCALE_POS), 'unit': 'm', 'use_for': 'centered positions'},
    'tracking_error_limit_m': {'value': float(cfg.MAX_POSITION_ERROR), 'unit': 'm', 'use_for': 'pos_error / tracking_error'},
    'tracking_failure_threshold_m': {'value': float(cfg.POS_ERROR_FAIL_THRESHOLD), 'unit': 'm', 'use_for': 'large tracking-error safety terms'},
    'reference_position_amp_m': {'value': float(cfg.REF_POS_AMP), 'unit': 'm', 'use_for': 'reference-style position terms'},
    'two_mm_m': {'value': 0.002, 'unit': 'm', 'use_for': 'small tracking deadbands or margins'},
    'five_mm_m': {'value': 0.005, 'unit': 'm', 'use_for': 'tracking tolerance bonuses'},
    'one_cm_m': {'value': 0.010, 'unit': 'm', 'use_for': 'moderate tracking tolerance'},

    # Velocity scales
    'obs_velocity_mps': {'value': float(cfg.OBS_SCALE_VEL), 'unit': 'm/s', 'use_for': 'v_m, v_s'},
    'velocity_error_mps': {'value': DEFAULT_VELOCITY_ERROR_SCALE_MPS, 'unit': 'm/s', 'use_for': 'velocity_error'},
    'max_geometric_velocity_mps': {'value': float(cfg.V_MAX_GEOM), 'unit': 'm/s', 'use_for': 'large velocity safety terms'},
    'reference_velocity_mps': {'value': float(2.0 * np.pi * cfg.REF_POS_FREQ * cfg.REF_POS_AMP), 'unit': 'm/s', 'use_for': 'reference-style velocities'},
    'force_velocity_mps': {'value': float(cfg.FORCE_INPUT_AMP / max(cfg.BETA, 1e-9)), 'unit': 'm/s', 'use_for': 'force-driven velocity scale'},

    # Power/transparency scales
    'power_error_practical_w': {'value': float(cfg.MAX_POWER_ERROR), 'unit': 'W', 'use_for': 'transparency_error'},
    'power_error_theoretical_w': {'value': float(cfg.MAX_POWER_ERROR_THEORETICAL), 'unit': 'W', 'use_for': 'conservative transparency_error'},

    # Force scales
    'force_input_amp_n': {'value': float(cfg.FORCE_INPUT_AMP), 'unit': 'N', 'use_for': 'F_h sinusoid amplitude'},
    'human_force_est_n': {'value': float(cfg.F_H_SCALE_EST), 'unit': 'N', 'use_for': 'F_h'},
    'environment_force_theoretical_n': {'value': float(cfg.F_E_MAX_THEORETICAL), 'unit': 'N', 'use_for': 'F_e'},
    'force_difference_n': {'value': DEFAULT_FORCE_DIFF_SCALE_N, 'unit': 'N', 'use_for': 'force_diff'},

    # Pressure and flow scales
    'pressure_supply_pa': {'value': float(cfg.P_SUPPLY), 'unit': 'Pa', 'use_for': 'raw chamber pressures'},
    'pressure_atmosphere_pa': {'value': float(cfg.P_ATM), 'unit': 'Pa', 'use_for': 'absolute pressure offsets'},
    'pressure_difference_pa': {'value': float(cfg.OBS_SCALE_PRESSURE), 'unit': 'Pa', 'use_for': 'pressure differences'},
    'mass_flow_kg_s': {'value': float(cfg.OBS_SCALE_FLOW), 'unit': 'kg/s', 'use_for': 'mdot_L1, mdot_L2'},

    # Action and valve scales
    'action_voltage_v': {'value': DEFAULT_ACTION_SCALE_V, 'unit': 'V', 'use_for': 'u_v'},
    'action_delta_voltage_v': {'value': DEFAULT_ACTION_DELTA_SCALE_V, 'unit': 'V', 'use_for': 'action_delta / smoothness'},
    'valve_position': {'value': VALVE_POSITION_SCALE, 'unit': '-', 'use_for': 'x_v'},
    'valve_velocity': {'value': VALVE_VELOCITY_SCALE, 'unit': '1/s', 'use_for': 'x_v_dot'},
}

show_rows(REWARD_SOURCE_CATALOG, title='Reward sources you can use', max_rows=80)
show_rows([
    {'scale_name': name, **payload}
    for name, payload in REWARD_SCALE_CATALOG.items()
], title='Reward normalization scales you can use', max_rows=80)

REWARD_SHAPES = [
    {'shape': 'square', 'formula': '((source - target) / scale)^2'},
    {'shape': 'absolute', 'formula': 'abs((source - target) / scale)'},
    {'shape': 'deadband_square', 'formula': 'max(abs(source - target) - deadband, 0)^2 / scale^2'},
    {'shape': 'deadband_abs', 'formula': 'max(abs(source - target) - deadband, 0) / scale'},
    {'shape': 'above_threshold_square', 'formula': 'max(source - threshold, 0)^2 / scale^2'},
    {'shape': 'below_threshold_square', 'formula': 'max(threshold - source, 0)^2 / scale^2'},
    {'shape': 'tolerance_bonus', 'formula': 'max(0, 1 - abs(source - target) / margin)'},
    {'shape': 'gaussian_bonus', 'formula': 'exp(-0.5 * ((source - target) / scale)^2)'},
]
show_rows(REWARD_SHAPES, title='Reward shapes you can use', max_rows=20)

REWARD_TERMS = [
    {
        'name': 'tracking',
        'source': 'pos_error',
        'shape': 'square',
        'sign': 'penalty',
        'weight': 40.0,
        'scale_name': 'tracking_error_limit_m',
    },
    {
        'name': 'smooth_action',
        'source': 'action_delta',
        'shape': 'square',
        'sign': 'penalty',
        'weight': 0.05,
        'scale_name': 'action_delta_voltage_v',
    },
    # Example bonus: uncomment to reward very small tracking error directly.
    # {
    #     'name': 'near_zero_tracking_bonus',
    #     'source': 'pos_error',
    #     'shape': 'tolerance_bonus',
    #     'sign': 'bonus',
    #     'weight': 0.5,
    #     'target': 0.0,
    #     'margin_name': 'five_mm_m',
    #     'scale_name': 'five_mm_m',
    # },
    # Example pressure regularizer: uncomment to discourage large master pressure imbalance.
    # {
    #     'name': 'master_pressure_balance',
    #     'source': 'delta_P_m',
    #     'shape': 'absolute',
    #     'sign': 'penalty',
    #     'weight': 0.1,
    #     'scale_name': 'pressure_difference_pa',
    # },
]

REWARD_VARIANT = reward_variant_from_name('track_jerk_no_force_trans')
REWARD_SPEC = {
    'name': REWARD_VARIANT.name,
    'description': 'Built-in tracking + action-smoothness reward. Force and transparency penalties are removed.',
    'builtin_variant': REWARD_VARIANT.name,
    'scale_catalog': REWARD_SCALE_CATALOG,
    'terms': REWARD_TERMS,
    'penalties': {
        'stroke_limit': 250.0,
        'invalid_state': 100.0,
        'tracking_error_fail': 1000.0,
        'edge_buffer_m': 0.0,
        'low_force_threshold_n': 0.0,
    },
}

reward_rows = []
for term in REWARD_TERMS:
    reward_rows.append({
        'name': term['name'],
        'source': term['source'],
        'shape': term['shape'],
        'sign': term['sign'],
        'weight': term['weight'],
        'scale_name': term.get('scale_name', ''),
        'scale_value': REWARD_SCALE_CATALOG.get(term.get('scale_name', ''), {}).get('value', ''),
        'target': term.get('target', 0.0),
        'deadband': term.get('deadband_name', term.get('deadband', 0.0)),
        'threshold': term.get('threshold', 0.0),
        'margin': term.get('margin', ''),
    })
show_rows(reward_rows, title=f'Reward formula: {REWARD_VARIANT.name}', max_rows=40)


## 6. Reward Sanity Check

This cell evaluates the reward formula on hand-made scenarios before training. If a bad scenario scores better than a good one, fix the reward before launching PPO.


In [ ]:
def reward_context(pos_error_m, transparency_error_w, velocity_error_mps=0.0, force_diff_n=0.0, u_v=0.0, action_delta_v=0.0):
    f_h = 10.0
    f_e = f_h + force_diff_n
    v_s = 0.0
    v_m = velocity_error_mps
    x_s = 0.5 * float(cfg.L_CYL)
    x_m = x_s + pos_error_m
    context = {
        'time': 0.0,
        'time_fraction': 0.0,
        'env_id': 0.0,
        'x_m': x_m,
        'x_s': x_s,
        'x_m_centered': pos_error_m,
        'x_s_centered': 0.0,
        'v_m': v_m,
        'v_s': v_s,
        'P_m1': float(cfg.P_SUPPLY),
        'P_m2': float(cfg.P_SUPPLY),
        'P_s1': float(cfg.P_SUPPLY),
        'P_s2': float(cfg.P_SUPPLY),
        'delta_P_m': 0.0,
        'delta_P_s': 0.0,
        'P_m1_minus_P_s1': 0.0,
        'P_m2_minus_P_s2': 0.0,
        'mdot_L1': 0.0,
        'mdot_L2': 0.0,
        'x_v': 0.0,
        'x_v_dot': 0.0,
        'F_h': f_h,
        'F_e': f_e,
        'u_v': u_v,
        'requested_u_v': u_v,
        'action_delta': action_delta_v,
        'pos_error': pos_error_m,
        'tracking_error': pos_error_m,
        'velocity_error': velocity_error_mps,
        'transparency_error': transparency_error_w,
        'force_diff': force_diff_n,
        'edge_severity': 0.0,
        'low_force_edge_severity': 0.0,
    }
    for key, value in list(context.items()):
        context[f'abs_{key}'] = abs(float(value))
    return context

def reward_case(name, pos_error_m, transparency_error_w, velocity_error_mps=0.0, force_diff_n=0.0, u_v=0.0, action_delta_v=0.0):
    context = reward_context(
        pos_error_m,
        transparency_error_w,
        velocity_error_mps=velocity_error_mps,
        force_diff_n=force_diff_n,
        u_v=u_v,
        action_delta_v=action_delta_v,
    )
    if REWARD_VARIANT.formula_terms:
        reward, grouped_terms, custom_terms = reward_formula_from_context(context, REWARD_VARIANT)
    else:
        reward, track, transp, effort, jerk, velocity, force_diff = compute_reward_terms(
            pos_error=pos_error_m,
            velocity_error=velocity_error_mps,
            transparency_error=transparency_error_w,
            force_diff=force_diff_n,
            u_v=u_v,
            action_delta=action_delta_v,
            variant=REWARD_VARIANT,
        )
        grouped_terms = {
            'track': track,
            'transparency': transp,
            'effort': effort,
            'jerk': jerk,
            'velocity': velocity,
            'force_diff': force_diff,
        }
        custom_terms = {}
    row = {
        'scenario': name,
        'reward': round(float(reward), 4),
    }
    for key, value in grouped_terms.items():
        if abs(float(value)) > 1e-12:
            row[key] = round(float(value), 4)
    for key, value in custom_terms.items():
        row[f'term_{key}'] = round(float(value), 4)
    return row

sanity_rows = [
    reward_case('good tracking', 0.002, 1.0, u_v=0.5),
    reward_case('good tracking + changed transparency input', 0.002, 15.0, u_v=0.5),
    reward_case('poor tracking', 0.040, 1.0, u_v=0.5),
    reward_case('large action change', 0.002, 1.0, u_v=5.0, action_delta_v=5.0),
]
show_rows(sanity_rows, title='Reward sanity check', max_rows=10)


## 7. Build Run Command

This writes a documentation copy of the specs, builds the command, and shows the exact run configuration. Training uses the named built-in state and reward variants shown below.


In [ ]:
def signal_option(name, *, waveform, amp, bias, omega, phase):
    return {
        'name': str(name),
        'force_waveform': str(waveform),
        'force_amp': float(amp),
        'force_bias': float(bias),
        'force_freq_rad': float(omega),
        'force_phase': float(phase),
    }

TRAIN_SIGNAL_OPTIONS = []
for waveform in ['sine', 'multisine']:
    for amp in [3.0, 5.0, 7.0]:
        for bias in [12.0, 15.0, 18.0]:
            for omega in [6.0]:
                for phase in [0.0, 0.5 * np.pi, np.pi, 1.5 * np.pi]:
                    TRAIN_SIGNAL_OPTIONS.append(signal_option(
                        f'train_{waveform}_a{amp:g}_b{bias:g}_w{omega:g}_p{phase:.2f}',
                        waveform=waveform,
                        amp=amp,
                        bias=bias,
                        omega=omega,
                        phase=phase,
                    ))

EVAL_SIGNAL_OPTIONS = []
for waveform in ['sine', 'multisine']:
    for amp in [4.0, 6.0]:
        for bias in [13.5, 16.5]:
            for omega in [6.0]:
                for phase in [0.25 * np.pi, 0.75 * np.pi, 1.25 * np.pi, 1.75 * np.pi]:
                    EVAL_SIGNAL_OPTIONS.append(signal_option(
                        f'eval_{waveform}_a{amp:g}_b{bias:g}_w{omega:g}_p{phase:.2f}',
                        waveform=waveform,
                        amp=amp,
                        bias=bias,
                        omega=omega,
                        phase=phase,
                    ))
EVAL_SIGNAL_OPTIONS = EVAL_SIGNAL_OPTIONS[:int(CFG['test_episodes'])]

RUN_SPEC = {
    'cfg': CFG,
    'state': STATE_SPEC,
    'reward': REWARD_SPEC,
    'train_signals': TRAIN_SIGNAL_OPTIONS,
    'eval_signals': EVAL_SIGNAL_OPTIONS,
}
SPEC_HASH = short_hash(RUN_SPEC)
CFG['study_name'] = FINAL_RESULTS_SUBFOLDER
SPEC_BASENAME = f"{CFG['study_name']}_{SPEC_HASH}"

SPEC_DIR = PG_RESULTS / 'specs'
STATE_SPEC_PATH = SPEC_DIR / f"{SPEC_BASENAME}_state.json"
REWARD_SPEC_PATH = SPEC_DIR / f"{SPEC_BASENAME}_reward.json"
TRAIN_SIGNAL_SPEC_PATH = SPEC_DIR / f"{SPEC_BASENAME}_train_signals.json"
EVAL_SIGNAL_SPEC_PATH = SPEC_DIR / f"{SPEC_BASENAME}_eval_signals.json"
save_json(STATE_SPEC_PATH, STATE_SPEC)
save_json(REWARD_SPEC_PATH, REWARD_SPEC)
save_json(TRAIN_SIGNAL_SPEC_PATH, {'signals': TRAIN_SIGNAL_OPTIONS})
save_json(EVAL_SIGNAL_SPEC_PATH, {'signals': EVAL_SIGNAL_OPTIONS})

RUN_STUDY_NAME = CFG['study_name']
STEPS_PER_EPISODE = max(1, int(round(CFG['episode_duration_s'] / float(cfg.RL_DT))))
EPISODE_DERIVED_TIMESTEPS = int(CFG['train_episodes'] * STEPS_PER_EPISODE)
TRAIN_TIMESTEPS = int(CFG['total_timesteps'] or EPISODE_DERIVED_TIMESTEPS)
PPO_N_STEPS = int(CFG['ppo_n_steps'])
PPO_BATCH_SIZE = int(CFG['ppo_batch_size'])
PPO_N_EPOCHS = int(CFG['ppo_n_epochs'])
PPO_DEVICE = str(CFG['ppo_device'])
PPO_ROLLOUT_TIMESTEPS = int(CFG['parallel_envs'] * PPO_N_STEPS)
PPO_ACTUAL_TRAIN_TIMESTEPS = int(((TRAIN_TIMESTEPS + PPO_ROLLOUT_TIMESTEPS - 1) // PPO_ROLLOUT_TIMESTEPS) * PPO_ROLLOUT_TIMESTEPS)

RUN_ROOTS = {
    FE_KEY: policy_gradient_suite_root(FE_MODE, RUN_STUDY_NAME) / '00b' / RUN_DIR
}

CMD = [
    str(PYTHON),
    '-m',
    'TeleopWithRL.matlab_literal_env.policy_gradient_experiments.run_policy_gradient_experiments',
    '--algo', ALGO_KEY,
    '--study-name', RUN_STUDY_NAME,
    '--env-mode', CFG['env_mode'],
    '--episode-duration', str(CFG['episode_duration_s']),
    '--env-switch-time', str(CFG['env_switch_time_s']),
    '--fe-mode', FE_MODE,
    '--reset-position-mode', CFG['reset_position_mode'],
    '--stroke-limit-mode', CFG['stroke_limit_mode'],
    '--force-amp', str(CFG['force_amp_N']),
    '--force-bias', str(CFG['force_bias_N']),
    '--force-freq-rad', str(CFG['force_freq_rad_s']),
    '--force-phase', str(CFG['force_phase_rad']),
    '--force-waveform', CFG['force_waveform'],
    '--reward-variant', REWARD_VARIANT.name,
    '--state-variant', STATE_VARIANT.name,
    '--train-reset-options-json', str(TRAIN_SIGNAL_SPEC_PATH),
    '--eval-reset-options-json', str(EVAL_SIGNAL_SPEC_PATH),
    '--train-episodes', str(CFG['train_episodes']),
    '--total-timesteps', str(TRAIN_TIMESTEPS),
    '--parallel-envs', str(CFG['parallel_envs']),
    '--vec-env', CFG['vec_env'],
    '--ppo-n-steps', str(PPO_N_STEPS),
    '--ppo-batch-size', str(PPO_BATCH_SIZE),
    '--ppo-n-epochs', str(PPO_N_EPOCHS),
    '--ppo-device', PPO_DEVICE,
    '--eval-every-episodes', str(CFG['eval_every_episodes']),
    '--test-episodes', str(CFG['test_episodes']),
    '--seed', str(CFG['seed']),
    '--parallel-workers', str(CFG['parallel_workers']),
    '--worker-torch-threads', str(CFG['worker_torch_threads']),
]
if CFG['skip_existing']:
    CMD.append('--skip-existing')

show_rows(
    [{
        'algo': ALGO_LABEL,
        'study_name': RUN_STUDY_NAME,
        'final_results_subfolder': FINAL_RESULTS_SUBFOLDER,
        'spec_hash': SPEC_HASH,
        'reward_variant': REWARD_VARIANT.name,
        'state_variant': STATE_VARIANT.name,
        'state_features': ', '.join(STATE_VARIANT.feature_names),
        'episode_duration_s': CFG['episode_duration_s'],
        'env_switch_time_s': CFG['env_switch_time_s'],
        'stroke_limit_mode': CFG['stroke_limit_mode'],
        'force_amp_N': CFG['force_amp_N'],
        'force_bias_N': CFG['force_bias_N'],
        'force_freq_rad_s': CFG['force_freq_rad_s'],
        'train_signal_count': len(TRAIN_SIGNAL_OPTIONS),
        'eval_signal_count': len(EVAL_SIGNAL_OPTIONS),
        'train_episodes': CFG['train_episodes'],
        'steps_per_episode': STEPS_PER_EPISODE,
        'episode_derived_timesteps': EPISODE_DERIVED_TIMESTEPS,
        'train_timesteps': TRAIN_TIMESTEPS,
        'ppo_n_steps': PPO_N_STEPS,
        'ppo_batch_size': PPO_BATCH_SIZE,
        'ppo_n_epochs': PPO_N_EPOCHS,
        'ppo_device': PPO_DEVICE,
        'ppo_rollout_timesteps': PPO_ROLLOUT_TIMESTEPS,
        'ppo_actual_train_timesteps': PPO_ACTUAL_TRAIN_TIMESTEPS,
        'parallel_envs': CFG['parallel_envs'],
        'vec_env': CFG['vec_env'],
        'eval_every_episodes': CFG['eval_every_episodes'],
        'test_episodes': CFG['test_episodes'],
        'python_executable': str(PYTHON),
        'state_spec_json': str(STATE_SPEC_PATH),
        'reward_spec_json': str(REWARD_SPEC_PATH),
        'train_signal_json': str(TRAIN_SIGNAL_SPEC_PATH),
        'eval_signal_json': str(EVAL_SIGNAL_SPEC_PATH),
        'fe_mode': FE_MODE,
        'run_root': str(RUN_ROOTS[FE_KEY]),
        'model_dir': str(RUN_ROOTS[FE_KEY] / 'm'),
        'plots_dir': str(RUN_ROOTS[FE_KEY] / 'p'),
        'command': subprocess.list2cmdline(CMD),
    }],
    title=f'{ALGO_LABEL} run config',
    max_rows=10,
)
show_rows(TRAIN_SIGNAL_OPTIONS[:16], title='Training input signal pool preview', max_rows=16)
show_rows(EVAL_SIGNAL_OPTIONS[:16], title='Held-out evaluation signal preview', max_rows=16)


## 8. Run Training

This launches the switched-dynamics FE mode only. It can take a while for full `train_episodes`.


In [ ]:
print(subprocess.list2cmdline(CMD))
completed = subprocess.run(CMD, cwd=str(WORKSPACE), check=True)
print(f'Completed with return code {completed.returncode}.')


## 9. Focused Unified Evaluation Methodology

This section applies the frozen-policy evaluation battery used for the final PPO model. Evaluation is deterministic and separated from training: PPO can train stochastically, but every reported test below uses `policy.predict(..., deterministic=True)`.

The battery is intentionally one-factor-at-a-time:

```text
1. Nominal test
2. Human input force tests
   - amplitude variation
   - frequency variation
   - signal type variation
3. Environment tests
   - K_e variation
   - B_e variation
4. Initial-condition tests
5. Sudden release test
6. Empirical Bode test
```

The main non-Bode table reports RMS/peak tracking error, post-contact RMS/peak error, settling time, control energy, control smoothness, saturation percentage, and failure flags. The Bode table reports gain, gain dB, and phase lag versus frequency.


In [ ]:
from pathlib import Path

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import pandas as pd
except Exception:
    pd = None

from TeleopWithRL.matlab_literal_env.studies.focused_evaluation import (
    build_bode_scenarios,
    build_focused_scenarios,
    load_summary,
    run_focused_evaluation,
)

EVAL_RUN_ROOT = RUN_ROOTS[FE_KEY]
EVAL_MODEL_PATH = EVAL_RUN_ROOT / "m" / f"{RUN_DIR}_model.zip"
EVAL_SUMMARY_PATH = EVAL_RUN_ROOT / "l" / "summary.json"
FOCUSED_EVAL_DIR = EVAL_RUN_ROOT / "focused_eval_v1"

focused_eval_config = {
    "model_path": EVAL_MODEL_PATH,
    "summary_path": EVAL_SUMMARY_PATH,
    "out_dir": FOCUSED_EVAL_DIR,
    "deterministic": True,
    "include_bode": True,
    "save_plots": True,
    "seed": int(CFG["seed"]),
}

if EVAL_SUMMARY_PATH.exists():
    focused_summary = load_summary(EVAL_SUMMARY_PATH)
    focused_scenarios = build_focused_scenarios(focused_summary)
    focused_bode_scenarios = build_bode_scenarios(focused_summary)

    show_rows(
        [{
            "model_path": str(EVAL_MODEL_PATH),
            "summary_path": str(EVAL_SUMMARY_PATH),
            "output_dir": str(FOCUSED_EVAL_DIR),
            "normal_scenarios": len(focused_scenarios),
            "bode_scenarios": len(focused_bode_scenarios),
            "deterministic": focused_eval_config["deterministic"],
        }],
        title="Focused evaluation config",
        max_rows=5,
    )
    show_rows(
        [
            {
                "scenario": s.name,
                "group": s.group,
                "waveform": s.force_waveform,
                "A_N": s.force_amp,
                "omega_rad_s": s.force_freq_rad,
                "K_e": s.post_switch_Ke,
                "B_e": s.post_switch_Be,
                "initial_condition": s.initial_state_delta is not None,
                "release_time_s": s.force_release_time,
            }
            for s in focused_scenarios
        ],
        title="Focused non-Bode scenario battery",
        max_rows=80,
    )
    show_rows(
        [
            {"scenario": s.name, "omega_rad_s": s.force_freq_rad, "frequency_hz": s.force_freq_rad / (2.0 * np.pi)}
            for s in focused_bode_scenarios
        ],
        title="Empirical Bode scenarios",
        max_rows=20,
    )
else:
    print(f"Run summary not found yet: {EVAL_SUMMARY_PATH}")
    print("Train the PPO model first, then rerun this cell.")


## 10. Run Focus Evaluation

Run this after the trained PPO model exists. The cell writes the evaluation CSVs and plots under `focused_eval_v1` inside the PPO run folder, then displays the resulting tables and key figures.


In [ ]:
if not EVAL_MODEL_PATH.exists():
    raise FileNotFoundError(f"PPO model not found: {EVAL_MODEL_PATH}")

focused_result = run_focused_evaluation(
    model_path=focused_eval_config["model_path"],
    out_dir=focused_eval_config["out_dir"],
    seed=focused_eval_config["seed"],
    deterministic=focused_eval_config["deterministic"],
    include_bode=focused_eval_config["include_bode"],
    save_plots=focused_eval_config["save_plots"],
)

metrics_path = FOCUSED_EVAL_DIR / "focused_eval_metrics.csv"
bode_path = FOCUSED_EVAL_DIR / "focused_eval_bode.csv"
summary_path = FOCUSED_EVAL_DIR / "focused_eval_summary.json"
bode_plot_path = FOCUSED_EVAL_DIR / "plots" / "empirical_bode.png"
nominal_plot_path = FOCUSED_EVAL_DIR / "plots" / "scenarios" / "nominal.png"
pulse_plot_path = FOCUSED_EVAL_DIR / "plots" / "scenarios" / "signal_pulse.png"
release_plot_path = FOCUSED_EVAL_DIR / "plots" / "scenarios" / "sudden_release_t15s.png"

print(f"Focused evaluation data: {FOCUSED_EVAL_DIR}")
print(f"Metrics CSV: {metrics_path}")
print(f"Bode CSV: {bode_path}")
print(f"Summary JSON: {summary_path}")

if pd is not None:
    metrics_df = pd.read_csv(metrics_path)
    bode_df = pd.read_csv(bode_path)
    display(metrics_df)
    display(bode_df)
    failures = int((metrics_df["failure_flag"] != 0).sum())
    worst_rms = metrics_df.sort_values("rms_error_m", ascending=False).iloc[0]
    worst_post = metrics_df.sort_values("post_contact_rms_error_m", ascending=False).iloc[0]
    print(f"Failure flags: {failures}")
    print(f"Worst RMS case: {worst_rms['scenario']} ({worst_rms['rms_error_m']:.6f} m)")
    print(f"Worst post-contact RMS case: {worst_post['scenario']} ({worst_post['post_contact_rms_error_m']:.6f} m)")
else:
    display(focused_result["metrics"])
    display(focused_result["bode"])

for image_path in [nominal_plot_path, pulse_plot_path, release_plot_path, bode_plot_path]:
    if image_path.exists():
        show_image(image_path)
    else:
        print(f"Missing plot: {image_path}")
